# Diniz Ch.4 — Noisy sinusoid prediction and residual filters

保留兩個 sinusoid、5-tap predictor、Sign-Error/Power-of-Two/NLMS 與 30-sample monic sign-LMS 範例。notch analysis 使用 1 − z⁻¹W(z)，與 predictor W(z) 分開。

來源：`Diniz-Chap4-Signal-Pred.ipynb`。所有資料與結果留在記憶體，圖表使用共用深色風格。

執行順序：設定 → 參數與資料 → 模型與訓練 → 圖表與數值摘要。

## 1. Setup

In [ ]:
from pathlib import Path
import sys
from functools import partial

import numpy as np

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "adsp").is_dir())
if str(project_root) not in sys.path:
  sys.path.insert(0, str(project_root))

from adsp import data, training, statistics as stats
from adsp import plotting as plot
from adsp import filters as af
from adsp import equalization as ae
from adsp.datasets import SampleBatch

plot.use_dark()

## 2. Parameters and factories

In [ ]:
n_samples, n_trials, seed = 1000, 50, 411
make_data = partial(data.sinusoid_prediction, n_samples, noise_var=1.0)
models = {
  "Sign-Error LMS": partial(af.SignErrorLMS, filter_order=4, mu=0.0028),
  "Power-of-Two LMS": partial(af.PowerOfTwoErrorLMS, filter_order=4, mu=0.0044, bd=12),
  "NLMS": partial(af.NLMS, filter_order=4, mu=0.4, tau=1e-6),
}
results = training.compare(models, make_data, n_trials=n_trials, seed=seed)

## 3. Learning, spectra and actual residual-filter response

In [ ]:
plot.learning(results, scale="log", window=21)
plot.show()
first = next(iter(results.values())).representative
plot.spectrum({"Observed input": first.x, **{name: res.representative.e for name, res in results.items()}})
plot.show()
residual_taps = {name: stats.prediction_error_taps(res.representative.w_final) for name, res in results.items()}
plot.frequency_response(residual_taps, title="Prediction-error filter response")
plot.show()
plot.zeros(residual_taps)
plot.show()
training.summary(results)

## 4. Monic sign-LMS residual filter

In [ ]:
x = np.sin(np.pi * np.arange(30) / 3)
batch = SampleBatch(x, np.zeros_like(x))
result = training.run(af.MonicSignLMS(n_taps=3, mu=0.01), batch, name="Monic sign-LMS")
plot.lines({"Sinusoid input": x, "Residual output": result.y}, title="30-sample monic-filter illustration")
plot.show()
plot.coefficients(result, title="Leading residual tap remains exactly one")
plot.show()